# IFC-Bereinigung nach IDS

Dieses Notebook prüft ein IFC-Modell gegen eine IDS, verschiebt Properties ins Pset «BFH_Minor» und entfernt Psets und Properties, die die IDS nicht verlangt.

**Ablauf**

1. Setup
2. Konfiguration (`IDS_URL`, `DRY_RUN`)
3. IDS laden, IFC hochladen und Prüfung vorher
4. Properties verschieben
5. Überzählige Psets bereinigen
6. Speichern, Nachprüfung, Download

**Sicherheitsnetz:** Solange `DRY_RUN = True` ist, wird das Modell nicht verändert. Die Zellen 4 und 5 listen nur die geplanten Änderungen. Erst mit `DRY_RUN = False` werden die Zellen 4 und 5 ausgeführt und danach in Zelle 6 gespeichert.

**Geltungsbereich:** Bearbeitet werden nur Bauteile, auf die eine Spezifikation der IDS zutrifft (Applicability, z. B. IfcBearing mit PredefinedType, IfcBridge). Berücksichtigt werden Property Sets (IfcPropertySet) am Bauteil selbst. Mengen (Qto), Typ-Psets und andere Bauteile bleiben unverändert. Gemeinsam genutzte Psets werden nicht gesondert behandelt.

In [ ]:
# Zelle 1: Setup
!pip install -q ifcopenshell ifctester

import ifcopenshell
print("ifcopenshell", ifcopenshell.version)

In [ ]:
# Zelle 2: Konfiguration

# Raw-URL der IDS-Datei im GitHub-Repo (Schaltfläche «Raw»)
IDS_URL = "https://raw.githubusercontent.com/Raef77/bxmid03-hs26-team-1/main/requirements/Team1_Lager_Elastomer.ids"

# True: nur planen, das Modell wird nicht verändert. Erst nach Prüfung der Pläne auf False setzen.
DRY_RUN = True

# Ab hier nichts ändern
IDS_PATH = "ids.ids"
IFC_AUSGABE = "bereinigt.ifc"
BERICHT_VORHER = "bericht_vorher.html"
BERICHT_NACHHER = "bericht_nachher.html"
print("IDS_URL:", IDS_URL)
print("DRY_RUN:", DRY_RUN)

In [ ]:
# Zelle 3: IDS laden, IFC hochladen, Prüfung vorher
import urllib.request, urllib.error
from google.colab import files
import ifcopenshell
import ifcopenshell.util.element as el_util
from ifctester import ids, reporter

if not IDS_URL.strip():
    raise ValueError("IDS_URL in Zelle 2 ist leer.")
try:
    urllib.request.urlretrieve(IDS_URL, IDS_PATH)
except urllib.error.HTTPError as e:
    raise RuntimeError(f"IDS-Download fehlgeschlagen (HTTP {e.code}). Abbruch.") from e


def pruefen(modell, html_pfad, titel):
    """Prüft das Modell gegen die IDS, schreibt den HTML-Bericht und gibt Kennzahlen zurück."""
    specs = ids.open(IDS_PATH)
    specs.validate(modell)
    html = reporter.Html(specs)
    html.report()
    html.to_file(html_pfad)
    spec_ok = sum(1 for s in specs.specifications if s.status is True)
    spec_nok = sum(1 for s in specs.specifications if s.status is False)
    req_ok = req_nok = 0
    for s in specs.specifications:
        for r in s.requirements:
            req_ok += len(getattr(r, "passed_entities", None) or [])
            req_nok += len(getattr(r, "failures", None) or [])
    for s in specs.specifications:
        print(f"  {s.identifier or s.name}: {'erfüllt' if s.status else 'NICHT erfüllt'} "
              f"({len(s.applicable_entities)} zutreffende Bauteile)")
    print(f"Prüfung {titel}: Spezifikationen erfüllt {spec_ok}, nicht erfüllt {spec_nok}")
    print(f"Prüfung {titel}: Anforderungen (je Bauteil) erfüllt {req_ok}, nicht erfüllt {req_nok}")
    print("Bericht:", html_pfad)
    return {"spec_ok": spec_ok, "spec_nok": spec_nok, "req_ok": req_ok, "req_nok": req_nok}


print("IDS geladen:", IDS_URL)
hochgeladen = files.upload()
IFC_NAME = next((n for n in hochgeladen if n.lower().endswith(".ifc")), None)
if IFC_NAME is None:
    raise FileNotFoundError("Keine .ifc-Datei hochgeladen.")
modell = ifcopenshell.open(IFC_NAME)
print("IFC geladen:", IFC_NAME, "| Schema:", modell.schema)

ERGEBNIS_VORHER = pruefen(modell, BERICHT_VORHER, "vorher")

In [ ]:
# Zelle 4: Properties ins Pset «BFH_Minor» verschieben
import ifcopenshell.api
import ifcopenshell.util.element as el_util
from xml.etree import ElementTree as ET

NS = {"i": "http://standards.buildingsmart.org/IDS"}


def _wert(knoten, pfad):
    n = knoten.find(pfad, NS)
    s = n.find("i:simpleValue", NS) if n is not None else None
    return s.text if s is not None else None


# Regeln aus der IDS lesen: Klasse, PredefinedType, verlangte (Pset, Property)
REGELN = []
for sp in ET.parse(IDS_PATH).getroot().iter("{%s}specification" % NS["i"]):
    ent = sp.find("i:applicability/i:entity", NS)
    if ent is None:
        continue
    REGELN.append({
        "klasse": _wert(ent, "i:name").upper(),
        "ptype": _wert(ent, "i:predefinedType"),
        "props": [(_wert(p, "i:propertySet"), _wert(p, "i:baseName"))
                  for p in sp.findall("i:requirements/i:property", NS)],
    })


def regeln_fuer(el):
    return [r for r in REGELN
            if el.is_a().upper() == r["klasse"]
            and (r["ptype"] is None or getattr(el, "PredefinedType", None) == r["ptype"])]


def eigene_psets(el):
    return el_util.get_psets(el, psets_only=True, should_inherit=False)


ELEMENTE = {}
for klasse in sorted({r["klasse"] for r in REGELN}):
    for el in modell.by_type(klasse):
        if regeln_fuer(el):
            ELEMENTE[el.id()] = el
print("Betroffene Bauteile:", len(ELEMENTE))

# Verschiebungen planen
MOVE_PLAN = {}
KONFLIKTE = []
for eid, el in ELEMENTE.items():
    soll = {}
    for r in regeln_fuer(el):
        for ps, name in r["props"]:
            soll[name] = ps
    psets = eigene_psets(el)
    for name, ziel in soll.items():
        quellen = {ps: d[name] for ps, d in psets.items() if ps != ziel and name in d}
        if not quellen:
            continue
        ziel_wert = psets.get(ziel, {}).get(name)
        werte = {repr(v) for v in quellen.values()}
        if ziel_wert is not None and werte != {repr(ziel_wert)}:
            KONFLIKTE.append((el, name, ziel, ziel_wert, quellen))
        elif ziel_wert is None and len(werte) > 1:
            KONFLIKTE.append((el, name, ziel, None, quellen))
        else:
            wert = ziel_wert if ziel_wert is not None else next(iter(quellen.values()))
            MOVE_PLAN.setdefault(eid, []).append(
                {"name": name, "ziel": ziel, "quellen": list(quellen), "wert": wert})

n_moves = sum(len(v) for v in MOVE_PLAN.values())
print(f"Geplante Verschiebungen: {n_moves} | Konflikte: {len(KONFLIKTE)}")
for eid, moves in MOVE_PLAN.items():
    el = ELEMENTE[eid]
    for m in moves:
        print(f"  {el.GlobalId} {el.is_a()}: {m['name']} = {m['wert']!r}  {', '.join(m['quellen'])} -> {m['ziel']}")
for el, name, ziel, ziel_wert, quellen in KONFLIKTE:
    print(f"  KONFLIKT {el.GlobalId} {el.is_a()}: {name} | Ziel {ziel}={ziel_wert!r} | Quellen {quellen}")


def verschiebe(el, m):
    psets = eigene_psets(el)
    if m["ziel"] in psets:
        ps = modell.by_id(psets[m["ziel"]]["id"])
    else:
        ps = ifcopenshell.api.run("pset.add_pset", modell, product=el, name=m["ziel"])
    ifcopenshell.api.run("pset.edit_pset", modell, pset=ps, properties={m["name"]: m["wert"]})
    for q in m["quellen"]:
        ifcopenshell.api.run("pset.edit_pset", modell,
                             pset=modell.by_id(psets[q]["id"]), properties={m["name"]: None})


if DRY_RUN:
    print("\nDRY_RUN = True: Modell nicht verändert.")
else:
    for eid, moves in MOVE_PLAN.items():
        for m in moves:
            verschiebe(ELEMENTE[eid], m)
    print(f"\nDRY_RUN = False: {n_moves} Verschiebungen ausgeführt. Konflikte bleiben unverändert.")

In [ ]:
# Zelle 5: In der IDS nicht verlangte Psets und Properties bereinigen


def simuliere(psets, moves):
    """Zustand nach den geplanten Verschiebungen (nur für DRY_RUN)."""
    s = {k: dict(d) for k, d in psets.items()}
    for m in moves:
        for q in m["quellen"]:
            s.get(q, {}).pop(m["name"], None)
        s.setdefault(m["ziel"], {"id": None})[m["name"]] = m["wert"]
    return s


LOESCH_PSETS = []   # (Element-ID, Pset, Anzahl Properties)
LOESCH_PROPS = []   # (Element-ID, Pset, Property)
BEHALTEN = []       # Psets mit offenem Konflikt: bleiben unverändert
for eid, el in ELEMENTE.items():
    erlaubt = {}
    for r in regeln_fuer(el):
        for ps, name in r["props"]:
            erlaubt.setdefault(ps, set()).add(name)
    psets = eigene_psets(el)
    if DRY_RUN:
        psets = simuliere(psets, MOVE_PLAN.get(eid, []))
    geschuetzt = {q for (e, _n, _z, _zw, qs) in KONFLIKTE if e.id() == eid for q in qs}
    for ps, d in psets.items():
        props = [n for n in d if n != "id"]
        if ps in geschuetzt:
            BEHALTEN.append((eid, ps))
        elif ps not in erlaubt:
            LOESCH_PSETS.append((eid, ps, len(props)))
        else:
            for n in props:
                if n not in erlaubt[ps]:
                    LOESCH_PROPS.append((eid, ps, n))

n_props_in_psets = sum(n for _, _, n in LOESCH_PSETS)
print(f"Zu löschen: {len(LOESCH_PSETS)} Psets (mit {n_props_in_psets} Properties) "
      f"und {len(LOESCH_PROPS)} einzelne Properties in erlaubten Psets")
for eid, ps, n in LOESCH_PSETS:
    print(f"  {ELEMENTE[eid].GlobalId} Pset {ps} ({n} Properties)")
for eid, ps, n in LOESCH_PROPS:
    print(f"  {ELEMENTE[eid].GlobalId} Property {ps}.{n}")
for eid, ps in BEHALTEN:
    print(f"  BEHALTEN (Konflikt) {ELEMENTE[eid].GlobalId} Pset {ps}")

if DRY_RUN:
    print("\nDRY_RUN = True: Modell nicht verändert.")
else:
    for eid, ps, _ in LOESCH_PSETS:
        el = ELEMENTE[eid]
        aktuell = eigene_psets(el)
        if ps in aktuell:
            ifcopenshell.api.run("pset.remove_pset", modell, product=el,
                                 pset=modell.by_id(aktuell[ps]["id"]))
    for eid, ps, n in LOESCH_PROPS:
        el = ELEMENTE[eid]
        aktuell = eigene_psets(el)
        if ps in aktuell:
            ifcopenshell.api.run("pset.edit_pset", modell,
                                 pset=modell.by_id(aktuell[ps]["id"]), properties={n: None})
    print("\nDRY_RUN = False: Löschungen ausgeführt.")

In [ ]:
# Zelle 6: Speichern, Nachprüfung, Download
if DRY_RUN:
    print("DRY_RUN ist True: nichts gespeichert. Zuerst DRY_RUN = False setzen und die Zellen 4 und 5 erneut ausführen.")
else:
    modell.write(IFC_AUSGABE)
    nach = ifcopenshell.open(IFC_AUSGABE)
    ERGEBNIS_NACHHER = pruefen(nach, BERICHT_NACHHER, "nachher")
    print("\nVergleich vorher -> nachher")
    for k in ("spec_ok", "spec_nok", "req_ok", "req_nok"):
        print(f"  {k}: {ERGEBNIS_VORHER[k]} -> {ERGEBNIS_NACHHER[k]}")
    print("Offene Konflikte:", len(KONFLIKTE))
    for f in (IFC_AUSGABE, BERICHT_VORHER, BERICHT_NACHHER):
        files.download(f)